# 08 — Grad-CAM (Figure 8)

**Tujuan.** Grad-CAM post-hoc pada Coordinate Attention dengan checkpoint eksperimen yang terverifikasi. Default: APTOS2019 `balanced_v1/aptos_r16`, preprocessing V1, split validation. Figure 8 tidak menentukan target layer; `blocks.5.attention` adalah asumsi. Panel overlay memakai piksel input model setelah crop/resize agar koordinat tepat. Tidak ada training, optimizer step, atau final-test inference. Output sintetis bukan bukti lesi.

In [1]:
from pathlib import Path
import os, sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "reproduction/cadvit/model.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
ARTIFACT_ROOT = Path(os.environ["CAD_VIT_ARTIFACT_ROOT"]).expanduser().resolve() if os.environ.get("CAD_VIT_ARTIFACT_ROOT") else None
print("Artifact root:", ARTIFACT_ROOT if ARTIFACT_ROOT else "PENDING (set CAD_VIT_ARTIFACT_ROOT)")

Artifact root: PENDING (set CAD_VIT_ARTIFACT_ROOT)


In [2]:
import torch
from reproduction.cadvit.model import CADViTCandidate
from reproduction.cadvit.interpretability import gradcam, DEFAULT_TARGET_LAYER
torch.set_num_threads(1)
synthetic_model = CADViTCandidate().eval()
synthetic_result = gradcam(synthetic_model, torch.randn(1, 3, 64, 64))
print("SYNTHETIC_TEST target", DEFAULT_TARGET_LAYER,
      "heatmap", tuple(synthetic_result["heatmap"].shape),
      "range", (float(synthetic_result["heatmap"].min()), float(synthetic_result["heatmap"].max())))
print("PENDING real train/validation RGB + trusted checkpoint; no test image is opened")

SYNTHETIC_TEST target blocks.5.attention heatmap (64, 64) range (0.0, 0.0)
PENDING real train/validation RGB + trusted checkpoint; no test image is opened


In [ ]:
import csv
import hashlib
import numpy as np
from PIL import Image
from IPython.display import display
import matplotlib.pyplot as plt
from reproduction.cadvit.interpretability import load_controlled_best, plot_gradcam
from reproduction.cadvit.visualization import guarded_non_test_image, save_campaign_figure
from reproduction.cadvit.experiments import artifact_paths, VARIANTS
from reproduction.cadvit.evaluation import file_sha256
from reproduction.cadvit.preprocessing import preprocess_fundus

if os.environ.get("CAD_VIT_RUN_GRADCAM") == "1":
    if ARTIFACT_ROOT is None:
        raise ValueError("Set CAD_VIT_ARTIFACT_ROOT to the verified external artifact root")
    campaign = os.environ.get("CAD_VIT_CAMPAIGN", "balanced_v1")
    variant = os.environ.get("CAD_VIT_VARIANT", "aptos_r16")
    source_split = os.environ.get("CAD_VIT_AUDIT_SPLIT", "val")
    if (campaign, variant) != ("balanced_v1", "aptos_r16"):
        raise ValueError("Only the existing balanced APTOS V1 protocol is supported here; historical V2 and DDR recovery require separate provenance verification")
    if source_split not in {"train", "val"}:
        raise ValueError("Only train/val permitted")
    paths = artifact_paths(ARTIFACT_ROOT, VARIANTS[variant],
                           data_root=os.environ.get("CAD_VIT_DATA_ROOT"), campaign=campaign)
    dataset_root = Path(paths["dataset_root"]).resolve()
    manifest_path = Path(os.environ.get("CAD_VIT_AUDIT_MANIFEST", str(paths["manifest"])))
    for required in (manifest_path, paths["run_dir"] / "best.pt", paths["run_dir"] / "config.json"):
        if not required.is_file():
            raise FileNotFoundError(required)
    model, provenance = load_controlled_best(ARTIFACT_ROOT, campaign, variant, trust_checkpoint=True)
    if file_sha256(manifest_path) != provenance["manifest_sha256"]:
        raise ValueError("Audit manifest and checkpoint fingerprint differ")
    with manifest_path.open("r", encoding="utf-8-sig", newline="") as stream:
        rows = [r for r in csv.DictReader(stream) if r["split"] == source_split]
    explicit_image = os.environ.get("CAD_VIT_AUDIT_IMAGE")
    if explicit_image:
        selected = [Path(explicit_image)]
    else:
        # Predefined deterministic selection: one image per actual grade, not selected by heatmap or prediction.
        selected = []
        for grade in range(5):
            group = [r for r in rows if int(r["label"]) == grade]
            if not group:
                print("Missing grade in selected split:", grade)
                continue
            row = min(group, key=lambda r: (hashlib.sha256(("42:" + r["image_id"]).encode()).hexdigest(), r["image_id"]))
            selected.append(dataset_root / row["image_path"])
    if not selected:
        raise ValueError("No verified non-test candidate images")
    device = torch.device(os.environ.get("CAD_VIT_GRADCAM_DEVICE", "cpu"))
    model = model.to(device).eval()
    for image_path in selected:
        checked_path, row = guarded_non_test_image(image_path, dataset_root, manifest_path, source_split)
        with Image.open(checked_path) as image:
            pixels = preprocess_fundus(image)
        tensor = torch.from_numpy(np.array(pixels, copy=True)).permute(2, 0, 1).float()[None].to(device) / 255
        result = gradcam(model, tensor)
        heatmap = result["heatmap"]
        if not torch.isfinite(heatmap).all():
            raise RuntimeError("Non-finite heatmap")
        print("image_id", row["image_id"], "split", source_split, "label", row["label"],
              "prediction", result["predicted_class"], "target", result["target_class"],
              "confidence", result["confidence"], "heatmap range", (float(heatmap.min()), float(heatmap.max())))
        if float(heatmap.max()) == float(heatmap.min()):
            print("WARNING: constant heatmap; no spatial attribution demonstrated")
        print("Provenance:", provenance)
        figure = plot_gradcam(pixels, result, int(row["label"]), provenance)
        display(figure)
        if os.environ.get("CAD_VIT_SAVE_FIGURES") == "1":
            stem = "gradcam_" + "".join(ch if ch.isalnum() else "_" for ch in row["image_id"])
            print("Saved", save_campaign_figure(figure, ARTIFACT_ROOT, campaign, variant, stem))
        plt.close(figure)
else:
    print("PENDING real Grad-CAM: set CAD_VIT_RUN_GRADCAM=1 only after verifying existing artifacts")

**Eksekusi real-image.** Jalankan notebook di runtime yang memiliki dataset dan checkpoint asli. Set `CAD_VIT_ARTIFACT_ROOT` ke direktori artefak eksternal yang sudah ada; `CAD_VIT_RUN_GRADCAM=1` mengaktifkan real-image. `CAD_VIT_DATA_ROOT` opsional, harus berupa root dataset APTOS yang cocok dengan path relatif manifest; `CAD_VIT_AUDIT_MANIFEST` opsional. Default `val`, satu contoh deterministik per grade (seed string 42), tanpa memilih hasil berdasarkan prediksi atau heatmap. `CAD_VIT_AUDIT_IMAGE` opsional untuk satu gambar train/val. `CAD_VIT_GRADCAM_DEVICE` default CPU; pilih CUDA hanya jika GPU tersedia dan disetujui. `CAD_VIT_SAVE_FIGURES=1` opsional untuk menyimpan PNG pada direktori figures kampanye yang sudah digunakan proyek; default tidak menulis PNG eksternal. Simpan output notebook hanya setelah eksekusi benar-benar berhasil dan cek izin publikasi citra. Jangan menimpa hasil historis. Visualisasi adalah atribusi model, bukan validasi klinis lesi. Hasil real-image tetap belum diverifikasi sampai runtime asli dijalankan.